# Single Task Adaptation Visualization for MLP MAML (CAD_TEST)

This notebook visualizes how MLP MAML model performs adaptation on a single task from CAD_TEST AND cells.

**Task Definition:**
- Task = (cell_name, delay_type, related_pin, slew_idx, load_idx)
- Support: 10 samples from 10 PVT conditions at same (slew, load)
- Query: 5 samples from 5 test PVT conditions at same (slew, load)

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import random
import re
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict, defaultdict

# Add paths
sys.path.insert(0, '/home/tkdgn2907/Deepsets_test/MAML/Projects/model_code')
sys.path.insert(0, '/home/tkdgn2907/Deepsets_test/MAML/Projects/pretraining/model_test_code/utils')
sys.path.insert(0, '/home/tkdgn2907/Deepsets_test/MAML/Projects/pretraining/model_test_code/check_point')

from maml_optimized import OptimizedMAML, MAMLModel_3hidden

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Data configuration
DATA_DIR = '/home/tkdgn2907/Deepsets_test/MAML/Projects/CAD_TEST/AND_cells_extracted'
DATA_TYPE = 'cell'  # 'cell' or 'transition'

# Model configuration (same as MAML_topology_validation.py)
MODEL_PATH = '/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/training_loss_taskdivide_all/cell_innerdiv100_meta32_combined_519traintask_full1DMAML_weights_3hidden_(40)_300000_inner1_upgraded_tsmc.pth'

# GPU settings
GPU_ID = '0'

# Task selection (set to None for random selection)
RANDOM_TASK_ID = None  # e.g., 100, or None for random

# Adaptation settings
ADAPTATION_METHOD = 'selective_adam'  # 'selective_adam' or 'adam'
MAX_ADAM_STEPS = 200  # For loss curve analysis
DEFAULT_ADAM_STEPS = 40  # Default number of steps

# Test/Support split
TEST_CONDITIONS = ['ff0p88v125c', 'ss0p72v125c', 'ff0p99vm40c', 'ss0p81vm40c', 'tt1p0v25c']

# TSMC Process Parameters
PARAM_A = [1.427, 1.457, 1.430, 1.470, 1.443, 1.483, 1.43, 1.47, 1.43, 1.47]
PARAM_B = [0.026, 0.045, 0, 0, -0.026, -0.05, 0.0208, -0.04, 0.036, -0.0208]
PARAM_C = [0.024, 2.000, 0.024, 2.000, 0.024, 2.000, 0.024, 2.000, 0.024, 2.000]
CORNER_TO_IDX = {'FF': 0, 'TT': 1, 'SS': 2, 'FS': 3, 'SF': 4}

In [ ]:
# GPU settings
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'Current cuda device: {torch.cuda.current_device()}')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_filename(filename):
    """Parse lib filename to extract corner, voltage, temperature."""
    match = re.search(r'(ff|tt|ss|fs|sf)(\d+p\d+)v(m?\d+)c', filename.lower())
    if not match:
        raise ValueError(f"Cannot parse filename: {filename}")
    
    corner = match.group(1).upper()
    voltage_str = match.group(2).replace('p', '.')
    voltage = float(voltage_str)
    
    temp_str = match.group(3)
    if temp_str.startswith('m'):
        temperature = -float(temp_str[1:])
    else:
        temperature = float(temp_str)
    
    return corner, voltage, temperature


def get_abc_params(corner):
    """Get a, b, c parameters for given corner."""
    idx = CORNER_TO_IDX.get(corner.upper(), 1)
    nmos_idx = idx * 2
    pmos_idx = idx * 2 + 1
    return {
        'a_n': PARAM_A[nmos_idx], 'a_p': PARAM_A[pmos_idx],
        'b_n': PARAM_B[nmos_idx], 'b_p': PARAM_B[pmos_idx],
        'c_n': PARAM_C[nmos_idx], 'c_p': PARAM_C[pmos_idx],
    }


def compute_abc_for_and_cell(abc_params, delay_indicator):
    """Compute a, b, c for AND cells."""
    a_param = (abc_params['a_n'] + abc_params['a_p']) / 2
    b_param = abc_params['b_n'] + abc_params['b_p']
    c_param = abc_params['c_n'] + abc_params['c_p']
    return a_param, b_param, c_param


def parse_lib_file(lib_path):
    """Parse .tlib file and extract timing data."""
    with open(lib_path, 'r') as f:
        content = f.read()
    
    samples = []
    cell_pattern = r'cell\s*\((\w+)\)\s*\{'
    
    for cell_match in re.finditer(cell_pattern, content):
        cell_name = cell_match.group(1)
        cell_start = cell_match.end()
        
        brace_count = 1
        cell_end = cell_start
        for i in range(cell_start, len(content)):
            if content[i] == '{':
                brace_count += 1
            elif content[i] == '}':
                brace_count -= 1
                if brace_count == 0:
                    cell_end = i
                    break
        
        cell_content = content[cell_start:cell_end]
        timing_pattern = r'timing\s*\(\)\s*\{\s*related_pin\s*:\s*"(\w+)"'
        
        for timing_match in re.finditer(timing_pattern, cell_content):
            related_pin = timing_match.group(1)
            timing_start = timing_match.end()
            
            t_brace_count = 1
            timing_end = timing_start
            for i in range(timing_start, len(cell_content)):
                if cell_content[i] == '{':
                    t_brace_count += 1
                elif cell_content[i] == '}':
                    t_brace_count -= 1
                    if t_brace_count == 0:
                        timing_end = i
                        break
            
            timing_content = cell_content[timing_start:timing_end]
            
            for delay_type in ['cell_rise', 'cell_fall', 'rise_transition', 'fall_transition']:
                table_pattern = rf'{delay_type}\s*\([^)]+\)\s*\{{'
                table_match = re.search(table_pattern, timing_content)
                
                if table_match:
                    table_start = table_match.end()
                    tb_count = 1
                    table_end = table_start
                    for i in range(table_start, len(timing_content)):
                        if timing_content[i] == '{':
                            tb_count += 1
                        elif timing_content[i] == '}':
                            tb_count -= 1
                            if tb_count == 0:
                                table_end = i
                                break
                    
                    table_content = timing_content[table_start:table_end]
                    
                    idx1_match = re.search(r'index_1\s*\(\s*"([^"]+)"\s*\)', table_content)
                    idx2_match = re.search(r'index_2\s*\(\s*"([^"]+)"\s*\)', table_content)
                    values_match = re.search(r'values\s*\(\s*(.*?)\s*\)\s*;', table_content, re.DOTALL)
                    
                    if idx1_match and idx2_match and values_match:
                        index_1 = [float(x.strip()) for x in idx1_match.group(1).split(',')]
                        index_2 = [float(x.strip()) for x in idx2_match.group(1).split(',')]
                        
                        values_str = values_match.group(1).replace('\\', '').replace('\n', ' ')
                        rows = re.findall(r'"([^"]+)"', values_str)
                        values = [[float(x.strip()) for x in row.split(',')] for row in rows]
                        
                        samples.append({
                            'cell_name': cell_name,
                            'delay_type': delay_type,
                            'related_pin': related_pin,
                            'index_1': index_1,
                            'index_2': index_2,
                            'values': values,
                        })
    return samples


def create_mlp_input(samples, corner, voltage, temperature, data_type='cell'):
    """Create MLP input tensor from parsed samples."""
    abc_params = get_abc_params(corner)
    inputs, outputs, metadata = [], [], []
    
    target_types = ['cell_rise', 'cell_fall'] if data_type == 'cell' else ['rise_transition', 'fall_transition']
    
    for sample in samples:
        if sample['delay_type'] not in target_types:
            continue
        
        delay_indicator = -1 if 'rise' in sample['delay_type'] else 1
        a_param, b_param, c_param = compute_abc_for_and_cell(abc_params, delay_indicator)
        additional_dim = 2
        
        for row_idx, slew in enumerate(sample['index_1']):
            for col_idx, load in enumerate(sample['index_2']):
                if row_idx < len(sample['values']) and col_idx < len(sample['values'][row_idx]):
                    value = sample['values'][row_idx][col_idx]
                    input_vec = [a_param, b_param, c_param, temperature, voltage,
                                 additional_dim, delay_indicator, slew, load]
                    inputs.append(input_vec)
                    outputs.append([value])
                    metadata.append({
                        'cell_name': sample['cell_name'],
                        'delay_type': sample['delay_type'],
                        'related_pin': sample['related_pin'],
                        'slew': slew, 'load': load,
                        'slew_idx': row_idx, 'load_idx': col_idx,
                    })
    
    if len(inputs) == 0:
        return torch.tensor([]), torch.tensor([]), []
    return torch.tensor(inputs, dtype=torch.float32), torch.tensor(outputs, dtype=torch.float32), metadata

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

print("Loading data...")
base_path = Path(DATA_DIR)

test_data = {}
support_data = {}

for voltage_dir in ['0p8v', '0p9v', '1p0v']:
    dir_path = base_path / voltage_dir
    if not dir_path.exists():
        continue
    
    for lib_file in sorted(dir_path.glob('AND_*.tlib')):
        match = re.search(r'AND_lib1_(\w+)_base_400\.tlib', lib_file.name)
        if not match:
            continue
        
        condition = match.group(1)
        corner, voltage, temperature = parse_filename(lib_file.name)
        
        print(f"Loading {condition} ({corner}, {voltage}V, {temperature}C)...")
        samples = parse_lib_file(str(lib_file))
        inputs, outputs, metadata = create_mlp_input(samples, corner, voltage, temperature, DATA_TYPE)
        
        if len(inputs) == 0:
            continue
        
        if condition in TEST_CONDITIONS:
            test_data[condition] = (inputs, outputs, metadata)
        else:
            support_data[condition] = (inputs, outputs, metadata)

print(f"\nTest conditions ({len(test_data)}): {list(test_data.keys())}")
print(f"Support conditions ({len(support_data)}): {list(support_data.keys())}")

In [ ]:
# ============================================================
# APPLY NORMALIZATION (TSMC style - only temp, voltage, slew, load)
# ============================================================

# Combine support data for normalization stats
all_support_inputs = torch.cat([inputs for inputs, _, _ in support_data.values()], dim=0)

norm_indices = [3, 4, 7, 8]  # temperature, voltage, slew, load
feature_names = {3: 'temperature', 4: 'voltage', 7: 'slew', 8: 'load'}
norm_stats = {}

print("Normalization (TSMC style - a,b,c not normalized):")
for idx in norm_indices:
    mean = all_support_inputs[:, idx].mean().item()
    std = all_support_inputs[:, idx].std().item()
    if std == 0:
        std = 1.0
    norm_stats[idx] = (mean, std)
    print(f"  {feature_names[idx]} (idx {idx}): mean={mean:.6f}, std={std:.6f}")

def apply_normalization(data, norm_stats):
    for idx, (mean, std) in norm_stats.items():
        if std > 0:
            data[:, idx] = (data[:, idx] - mean) / std

# Apply normalization
print("\nApplying normalization...")
for condition in support_data:
    inputs, outputs, metadata = support_data[condition]
    apply_normalization(inputs, norm_stats)

for condition in test_data:
    inputs, outputs, metadata = test_data[condition]
    apply_normalization(inputs, norm_stats)

In [ ]:
# ============================================================
# BUILD TASK INDEX
# ============================================================

def build_index(data_dict):
    """Build index: cell_key -> condition -> (slew_idx, load_idx) -> sample_idx"""
    index = defaultdict(lambda: defaultdict(dict))
    for condition, (inputs, outputs, metadata) in data_dict.items():
        for idx, meta in enumerate(metadata):
            cell_key = (meta['cell_name'], meta['delay_type'], meta['related_pin'])
            sl_key = (meta['slew_idx'], meta['load_idx'])
            index[cell_key][condition][sl_key] = idx
    return index

support_index = build_index(support_data)
test_index = build_index(test_data)

support_conditions = list(support_data.keys())
test_conditions = list(test_data.keys())

# Collect all valid task keys
all_task_keys = []
for cell_key in support_index:
    available_sl_keys = None
    for condition in support_conditions:
        if condition in support_index[cell_key]:
            condition_sl_keys = set(support_index[cell_key][condition].keys())
            if available_sl_keys is None:
                available_sl_keys = condition_sl_keys
            else:
                available_sl_keys = available_sl_keys & condition_sl_keys
    
    if available_sl_keys:
        for sl_key in available_sl_keys:
            exists_in_test = all(
                cell_key in test_index and 
                condition in test_index[cell_key] and 
                sl_key in test_index[cell_key][condition]
                for condition in test_conditions
            )
            if exists_in_test:
                all_task_keys.append((cell_key, sl_key))

print(f"Total valid tasks: {len(all_task_keys)}")

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================

print(f"Loading model: {MODEL_PATH}")

maml_model = OptimizedMAML(
    model=MAMLModel_3hidden(in_features=9, layer_length=40),
    dataset_in=None,
    dataset_out=None,
    inner_lr=0.001,
    meta_lr=0.0001
)

state_dict = torch.load(MODEL_PATH, map_location=device)
maml_model.model.load_state_dict(state_dict)
maml_model.model.to(device)
maml_model.model.eval()

print("Model loaded successfully!")

In [ ]:
# ============================================================
# SELECT TASK
# ============================================================

if RANDOM_TASK_ID is None:
    task_idx = random.randint(0, len(all_task_keys) - 1)
else:
    task_idx = RANDOM_TASK_ID

cell_key, sl_key = all_task_keys[task_idx]

print("=" * 80)
print(f"Selected Task: {task_idx} / {len(all_task_keys)}")
print("=" * 80)
print(f"  Cell name: {cell_key[0]}")
print(f"  Delay type: {cell_key[1]}")
print(f"  Related pin: {cell_key[2]}")
print(f"  Slew idx: {sl_key[0]}, Load idx: {sl_key[1]}")
print("=" * 80)

In [ ]:
# ============================================================
# BUILD SUPPORT AND QUERY SETS
# ============================================================

# Build support set: 10 samples from 10 support conditions
X_support_list, y_support_list = [], []
support_info = []

for condition in support_conditions:
    sample_idx = support_index[cell_key][condition][sl_key]
    inputs, outputs, metadata = support_data[condition]
    X_support_list.append(inputs[sample_idx:sample_idx+1])
    y_support_list.append(outputs[sample_idx:sample_idx+1])
    support_info.append({'condition': condition, 'output': outputs[sample_idx].item()})

# Build query set: 5 samples from 5 test conditions
X_query_list, y_query_list = [], []
query_info = []

for condition in test_conditions:
    sample_idx = test_index[cell_key][condition][sl_key]
    inputs, outputs, metadata = test_data[condition]
    X_query_list.append(inputs[sample_idx:sample_idx+1])
    y_query_list.append(outputs[sample_idx:sample_idx+1])
    query_info.append({'condition': condition, 'output': outputs[sample_idx].item()})

X_support = torch.cat(X_support_list, dim=0).to(device)
y_support = torch.cat(y_support_list, dim=0).to(device)
X_query = torch.cat(X_query_list, dim=0).to(device)
y_query = torch.cat(y_query_list, dim=0).to(device)

print(f"Support set: {X_support.shape[0]} samples from {len(support_conditions)} conditions")
print(f"Query set: {X_query.shape[0]} samples from {len(test_conditions)} conditions")

In [ ]:
# ============================================================
# DISPLAY TASK DATA
# ============================================================

print("\nSUPPORT DATA (10 PVT conditions):")
print("-" * 100)
print(f"{'Idx':<4} {'Condition':<15} {'Corner':<6} {'Voltage':<8} {'Temp':<8} {'Output (ns)':<12}")
print("-" * 100)

for i, info in enumerate(support_info):
    cond = info['condition']
    corner, voltage, temp = parse_filename(cond)
    print(f"{i:<4} {cond:<15} {corner:<6} {voltage:<8.2f} {temp:<8.0f} {info['output']:<12.6f}")

print("\nQUERY DATA (5 test conditions):")
print("-" * 100)
print(f"{'Idx':<4} {'Condition':<15} {'Corner':<6} {'Voltage':<8} {'Temp':<8} {'Output (ns)':<12}")
print("-" * 100)

for i, info in enumerate(query_info):
    cond = info['condition']
    corner, voltage, temp = parse_filename(cond)
    print(f"{i:<4} {cond:<15} {corner:<6} {voltage:<8.2f} {temp:<8.0f} {info['output']:<12.6f}")

In [ ]:
# ============================================================
# CALCULATE GRAD AND MOVE PARAMETERS
# ============================================================

y_mean = y_support.mean()
y_std = y_support.std()
y_norm = (y_support - y_mean) / y_std

# Get model predictions for scaling
with torch.no_grad():
    predictions_support = maml_model.model.model(X_support)
    min_val = predictions_support.min().item()
    max_val = predictions_support.max().item()

y_max = y_norm[:, 0].max().item()
y_min = y_norm[:, 0].min().item()

grad = (y_max - y_min) / (max_val - min_val) if abs(max_val - min_val) > 1e-8 else 1.0

# Calculate move (using tt0p9v25c as nominal - index 5)
nominal_idx = 5  # tt0p9v25c
center_input = X_support[nominal_idx:nominal_idx+1].clone()
center_input[0, 4] = 0.0  # Normalized voltage = 0
center = maml_model.model.model(center_input).item()
move = center - y_norm[nominal_idx, 0].item() / grad

print("Scaling Parameters:")
print(f"  y_mean: {y_mean.item():.6f}")
print(f"  y_std: {y_std.item():.6f}")
print(f"  y_norm range: [{y_min:.4f}, {y_max:.4f}]")
print(f"  Model pred range: [{min_val:.4f}, {max_val:.4f}]")
print(f"  grad: {grad:.6f}")
print(f"  move: {move:.6f}")
print(f"  center (nominal): {center:.6f}")

In [ ]:
# ============================================================
# MODEL PREDICTIONS (BEFORE ADAPTATION)
# ============================================================

with torch.no_grad():
    pred_query_before = maml_model.model.model(X_query)
    # Scale predictions to original space
    pred_query_before_scaled = (pred_query_before - move) * grad * y_std + y_mean

print("\nPREDICTIONS BEFORE ADAPTATION:")
print("-" * 90)
print(f"{'Idx':<4} {'Condition':<15} {'Actual (ns)':<15} {'Predicted (ns)':<15} {'Error (ns)':<12} {'Error (%)':<10}")
print("-" * 90)

for i, info in enumerate(query_info):
    actual = info['output']
    pred = pred_query_before_scaled[i, 0].item()
    error = pred - actual
    error_pct = abs(error / actual) * 100 if actual != 0 else 0
    print(f"{i:<4} {info['condition']:<15} {actual:<15.6f} {pred:<15.6f} {error:<12.6f} {error_pct:<10.2f}%")

In [ ]:
# ============================================================
# RUN ADAPTATION WITH LOSS TRACKING
# ============================================================

def run_adaptation_with_tracking(initial_model, X_support, y_support, X_query, y_query,
                                  max_steps=200, lr=3e-4):
    """Run adaptation and track loss at each step."""
    
    # Create a copy of the model
    model = nn.Sequential(OrderedDict([
        ('l1', nn.Linear(9, 40)),
        ('relu1', nn.ReLU()),
        ('l2', nn.Linear(40, 40)),
        ('relu3', nn.ReLU()),
        ('l4', nn.Linear(40, 40)),
        ('relu2', nn.ReLU()),
        ('l3', nn.Linear(40, 1))
    ])).to(device)
    model.load_state_dict(initial_model.state_dict())
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Normalize support data
    y_mean = y_support.mean()
    y_std = y_support.std()
    y_target = (y_support - y_mean) / y_std
    
    # Track metrics
    support_losses = []
    query_losses = []
    query_nrmses = []
    query_mapes = []
    predictions_over_time = []
    
    for step in range(max_steps + 1):
        with torch.no_grad():
            # Support loss
            pred = model(X_support)
            support_loss = criterion(pred, y_target).item()
            support_losses.append(support_loss)
            
            # Query metrics
            pred_query = model(X_query)
            pred_query_orig = pred_query * y_std + y_mean
            
            query_mse = ((pred_query_orig - y_query) ** 2).mean().item()
            query_losses.append(query_mse)
            
            query_range = y_query.max().item() - y_query.min().item()
            query_rmse = np.sqrt(query_mse)
            query_nrmse = (query_rmse / query_range * 100) if query_range > 0 else 0
            query_nrmses.append(query_nrmse)
            
            mape = (torch.abs(pred_query_orig - y_query) / (y_query + 1e-8)).mean().item() * 100
            query_mapes.append(mape)
            
            predictions_over_time.append(pred_query_orig.cpu().numpy().flatten())
        
        if step < max_steps:
            model.zero_grad()
            loss = criterion(model(X_support), y_target)
            loss.backward()
            optimizer.step()
    
    return {
        'support_losses': support_losses,
        'query_losses': query_losses,
        'query_nrmses': query_nrmses,
        'query_mapes': query_mapes,
        'predictions_over_time': predictions_over_time,
        'final_predictions': predictions_over_time[-1],
        'actual_values': y_query.cpu().numpy().flatten()
    }

print(f"Running adaptation for {MAX_ADAM_STEPS} steps...")
results = run_adaptation_with_tracking(
    maml_model.model.model, X_support, y_support, X_query, y_query,
    max_steps=MAX_ADAM_STEPS
)
print("Done!")

In [ ]:
# ============================================================
# ADAPTATION RESULTS OVER STEPS
# ============================================================

print("\nADAPTATION LOSS OVER STEPS:")
print("=" * 80)
print(f"{'Step':<8} {'Support Loss':<15} {'Query MSE':<15} {'Query NRMSE':<15} {'Query MAPE':<15}")
print("-" * 80)

key_steps = [0, 10, 20, 40, 60, 80, 100, 150, 200]
for step in key_steps:
    if step <= MAX_ADAM_STEPS:
        print(f"{step:<8} {results['support_losses'][step]:<15.6f} {results['query_losses'][step]:<15.6f} "
              f"{results['query_nrmses'][step]:<15.2f}% {results['query_mapes'][step]:<15.2f}%")

best_step = np.argmin(results['query_nrmses'])
print("-" * 80)
print(f"Best step: {best_step} with Query NRMSE: {results['query_nrmses'][best_step]:.2f}%")
print(f"Default step (40): Query NRMSE: {results['query_nrmses'][40]:.2f}%")

In [ ]:
# ============================================================
# VISUALIZATION: LOSS CURVES
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Support Loss
ax1 = axes[0, 0]
ax1.plot(results['support_losses'], 'b-', linewidth=1)
ax1.set_xlabel('Adaptation Step')
ax1.set_ylabel('Support Loss (MSE)')
ax1.set_title('Support Loss over Adaptation Steps')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=40, color='r', linestyle='--', alpha=0.5, label='Default (40 steps)')
ax1.axvline(x=best_step, color='g', linestyle='--', alpha=0.5, label=f'Best ({best_step} steps)')
ax1.legend()

# Query Loss
ax2 = axes[0, 1]
ax2.plot(results['query_losses'], 'r-', linewidth=1)
ax2.set_xlabel('Adaptation Step')
ax2.set_ylabel('Query Loss (MSE)')
ax2.set_title('Query Loss over Adaptation Steps')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=40, color='r', linestyle='--', alpha=0.5, label='Default (40 steps)')
ax2.axvline(x=best_step, color='g', linestyle='--', alpha=0.5, label=f'Best ({best_step} steps)')
ax2.legend()

# Query NRMSE
ax3 = axes[1, 0]
ax3.plot(results['query_nrmses'], 'g-', linewidth=1)
ax3.set_xlabel('Adaptation Step')
ax3.set_ylabel('Query NRMSE (%)')
ax3.set_title('Query NRMSE over Adaptation Steps')
ax3.grid(True, alpha=0.3)
ax3.axvline(x=40, color='r', linestyle='--', alpha=0.5, label='Default (40 steps)')
ax3.axvline(x=best_step, color='g', linestyle='--', alpha=0.5, label=f'Best ({best_step} steps)')
ax3.legend()

# Query MAPE
ax4 = axes[1, 1]
ax4.plot(results['query_mapes'], 'm-', linewidth=1)
ax4.set_xlabel('Adaptation Step')
ax4.set_ylabel('Query MAPE (%)')
ax4.set_title('Query MAPE over Adaptation Steps')
ax4.grid(True, alpha=0.3)
ax4.axvline(x=40, color='r', linestyle='--', alpha=0.5, label='Default (40 steps)')
ax4.axvline(x=best_step, color='g', linestyle='--', alpha=0.5, label=f'Best ({best_step} steps)')
ax4.legend()

plt.suptitle(f'Task: {cell_key[0]} / {cell_key[1]} / {cell_key[2]} / slew={sl_key[0]}, load={sl_key[1]}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: PREDICTIONS AT DIFFERENT STEPS
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

x_labels = [info['condition'] for info in query_info]
x_pos = np.arange(len(x_labels))

# Actual values
ax.plot(x_pos, results['actual_values'], 'ko-', markersize=10, linewidth=2, label='Actual')

# Predictions at different steps
steps_to_show = [0, 20, 40, 100, 200]
colors = ['red', 'orange', 'green', 'blue', 'purple']
markers = ['s', '^', 'D', 'v', 'p']

for step, color, marker in zip(steps_to_show, colors, markers):
    if step <= MAX_ADAM_STEPS:
        ax.plot(x_pos, results['predictions_over_time'][step], 
                f'{marker}--', color=color, markersize=8, linewidth=1, 
                alpha=0.7, label=f'Step {step}')

ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45, ha='right')
ax.set_xlabel('Test Condition')
ax.set_ylabel('Delay (ns)')
ax.set_title(f'Predictions at Different Adaptation Steps\nTask: {cell_key[0]} / {cell_key[1]}')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION: SUPPORT vs QUERY DATA (PVT CURVE)
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Sort by output value for better visualization
support_sorted = sorted(enumerate(support_info), key=lambda x: x[1]['output'])
query_sorted = sorted(enumerate(query_info), key=lambda x: x[1]['output'])

# Support data
support_x = np.arange(len(support_info))
support_y = [info['output'] for info in support_info]
ax.scatter(support_x, support_y, s=100, c='steelblue', marker='o', label='Support (10 PVT)', zorder=3)

# Query data
query_x = np.arange(len(support_info), len(support_info) + len(query_info))
query_y = [info['output'] for info in query_info]
ax.scatter(query_x, query_y, s=100, c='green', marker='s', label='Query Actual', zorder=3)

# Predictions at step 40
pred_40 = results['predictions_over_time'][40]
ax.scatter(query_x, pred_40, s=100, c='red', marker='^', label='Query Predicted (step 40)', zorder=3)

# Error lines
for i, (actual, pred) in enumerate(zip(query_y, pred_40)):
    ax.plot([query_x[i], query_x[i]], [actual, pred], 'r--', alpha=0.5, linewidth=1)

# Labels
all_labels = [info['condition'] for info in support_info] + [info['condition'] for info in query_info]
ax.set_xticks(np.arange(len(all_labels)))
ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=8)
ax.axvline(x=len(support_info) - 0.5, color='gray', linestyle=':', alpha=0.5)
ax.text(len(support_info)/2, ax.get_ylim()[1]*0.95, 'Support', ha='center', fontsize=10, color='steelblue')
ax.text(len(support_info) + len(query_info)/2, ax.get_ylim()[1]*0.95, 'Query', ha='center', fontsize=10, color='gray')

ax.set_xlabel('PVT Condition')
ax.set_ylabel('Delay (ns)')
ax.set_title(f'PVT Curve: 10 Support → 5 Query Prediction\nTask: {cell_key[0]} / {cell_key[1]} / {cell_key[2]}')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINAL RESULTS TABLE
# ============================================================

print("\n" + "=" * 90)
print("FINAL PREDICTION RESULTS (at step 40)")
print("=" * 90)
print(f"{'Idx':<4} {'Condition':<15} {'Actual (ns)':<15} {'Predicted (ns)':<15} {'Error (ns)':<12} {'Error (%)':<10}")
print("-" * 90)

pred_40 = results['predictions_over_time'][40]
actuals = results['actual_values']

for i, info in enumerate(query_info):
    actual = actuals[i]
    pred = pred_40[i]
    error = pred - actual
    error_pct = abs(error / actual) * 100 if actual != 0 else 0
    print(f"{i:<4} {info['condition']:<15} {actual:<15.6f} {pred:<15.6f} {error:<12.6f} {error_pct:<10.2f}%")

print("-" * 90)
rmse = np.sqrt(np.mean((pred_40 - actuals) ** 2))
range_val = actuals.max() - actuals.min()
nrmse = (rmse / range_val) * 100 if range_val > 0 else 0
mape = np.mean(np.abs((pred_40 - actuals) / (actuals + 1e-8))) * 100

print(f"\nTask Metrics:")
print(f"  RMSE: {rmse:.6f} ns")
print(f"  NRMSE: {nrmse:.2f}%")
print(f"  MAPE: {mape:.2f}%")

In [ ]:
# ============================================================
# COMPARISON: PREDICTIONS AT DIFFERENT STEPS
# ============================================================

print("\n" + "=" * 100)
print("PREDICTIONS AT DIFFERENT ADAPTATION STEPS")
print("=" * 100)
print(f"{'Condition':<15} {'Actual':<12} {'Step 0':<12} {'Step 20':<12} {'Step 40':<12} {'Step 100':<12} {'Step 200':<12}")
print("-" * 100)

steps = [0, 20, 40, 100, 200]
for i, info in enumerate(query_info):
    row = f"{info['condition']:<15} {actuals[i]:<12.4f}"
    for step in steps:
        if step <= MAX_ADAM_STEPS:
            row += f" {results['predictions_over_time'][step][i]:<12.4f}"
    print(row)

In [ ]:
# ============================================================
# ERROR ANALYSIS BY CONDITION TYPE
# ============================================================

pred_40 = results['predictions_over_time'][40]
errors = pred_40 - actuals
abs_errors = np.abs(errors)
pct_errors = np.abs(errors / (actuals + 1e-8)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute error per condition
ax1 = axes[0]
colors = ['green' if 'ff' in info['condition'] else 'red' if 'ss' in info['condition'] else 'blue' 
          for info in query_info]
bars = ax1.bar(range(len(query_info)), abs_errors, color=colors, alpha=0.7)
ax1.set_xticks(range(len(query_info)))
ax1.set_xticklabels([info['condition'] for info in query_info], rotation=45, ha='right')
ax1.set_ylabel('Absolute Error (ns)')
ax1.set_title('Absolute Error by Test Condition')
ax1.grid(True, alpha=0.3, axis='y')

# Add legend for corner types
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', alpha=0.7, label='FF (Fast)'),
                   Patch(facecolor='red', alpha=0.7, label='SS (Slow)'),
                   Patch(facecolor='blue', alpha=0.7, label='TT (Typical)')]
ax1.legend(handles=legend_elements)

# Percentage error per condition
ax2 = axes[1]
bars = ax2.bar(range(len(query_info)), pct_errors, color=colors, alpha=0.7)
ax2.set_xticks(range(len(query_info)))
ax2.set_xticklabels([info['condition'] for info in query_info], rotation=45, ha='right')
ax2.set_ylabel('Percentage Error (%)')
ax2.set_title('Percentage Error by Test Condition')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

# Print summary
print("\nError Summary by Corner:")
for corner in ['ff', 'ss', 'tt']:
    mask = [corner in info['condition'] for info in query_info]
    if any(mask):
        corner_errors = abs_errors[mask]
        corner_pct = pct_errors[mask]
        print(f"  {corner.upper()}: Mean Abs Error = {np.mean(corner_errors):.4f} ns, Mean % Error = {np.mean(corner_pct):.2f}%")